In [1]:
import os
os.chdir('../')
os.getcwd()

'/home/minh_khai/skin_disease/skin-disease'

## Load model

In [2]:
from abc import ABC, abstractmethod
from pathlib import Path

class BaseVisionModel(ABC):
    @abstractmethod
    def predict(self, imgs): ...
    
    @abstractmethod
    def save(self, path: Path) -> None: ...
    
    @abstractmethod
    def load(self, path: Path) -> None: ...
    
    @abstractmethod
    def save_onnx(self, path: Path, sample_input) -> None: ...

In [3]:
import torch.nn as nn
from torchvision.models import efficientnet_b0, resnet50

def build_efficientnet(num_classes: int, freeze_backbone: bool = True) -> nn.Module:
    model = efficientnet_b0(weights="IMAGENET1K_V1")
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    if freeze_backbone:
        for p in model.features.parameters():
            p.requires_grad = False
    return model

def build_resnet(num_classes: int, freeze_backbone: bool = True) -> nn.Module:
    model = resnet50(weights="IMAGENET1K_V1")
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    if freeze_backbone:
        for name, p in model.named_parameters():
            if not name.startswith("fc"):
                p.requires_grad = False
    return model

In [4]:
import torch
from pathlib import Path
import numpy as np
import onnxruntime as ort
import numpy as np
from pathlib import Path


def export_onnx(model, sample_input, output_path: Path, input_names=("input",), output_names=("output",), dynamic_axes=None):
    model.eval()
    torch.onnx.export(
        model,
        sample_input,
        str(output_path),
        input_names=list(input_names),
        output_names=list(output_names),
        dynamic_axes=dynamic_axes or {"input": {0: "batch_size"}, "output": {0: "batch_size"}},
        opset_version=17,
    )

class EfficientNetModel(BaseVisionModel):
    def __init__(self, num_classes: int, checkpoint_dir: Path = None):
        self.num_classes = num_classes
        self.checkpoint_dir = checkpoint_dir
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model  = build_efficientnet(num_classes, freeze_backbone=True).to(self.device)

    def predict(self, imgs):
        self.model.eval()
        with torch.no_grad():
            logits = self.model(imgs.to(self.device))
            probs = torch.softmax(logits, dim=1)
            confidence, pred_idx = torch.max(probs, dim=1)
        return pred_idx.item(), confidence.item()

    def save(self, path: Path) -> None:
        torch.save(self.model.state_dict(), path)

    def load(self, path: Path) -> None:
        checkpoint = torch.load(path, map_location=self.device)
        state = checkpoint["model_state"] if "model_state" in checkpoint else checkpoint
        self.model.load_state_dict(state)

    def save_onnx(self, path: Path, sample_input) -> None:
        export_onnx(self.model, sample_input, path)

2026-06-19 04:38:03.277818773 [W:onnxruntime:Default, device_discovery.cc:283 GetGpuDevices] Failed to detect devices under "/sys/class/drm/card0": device_discovery.cc:93 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"


In [5]:
from pathlib import Path

MODEL_REGISTRY = {
    "efficientnet": EfficientNetModel,
    # "resnet": ResNetModel,
}

class ModelFactory:
    @staticmethod
    def create(model_name: str, config, num_classes: int) -> BaseVisionModel:
        cls = MODEL_REGISTRY.get(model_name)
        if not cls:
            raise ValueError(f"No model registered for: '{model_name}'. Add it to MODEL_REGISTRY.")
        
        checkpoint_dir = Path(config.output_dir) / model_name
        return cls(num_classes, checkpoint_dir)

## Training

In [6]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class TrainMetrics:
    model_name:         str
    image_size:         int
    epochs:             int
    batch_size:         int
    lr:                 float
    unfreeze_epoch:     int
    weight_decay:       float
    top_k_checkpoints:  int
    early_stopping:     int

@dataclass(frozen=True)
class TrainConfig:
    train_dir:  Path
    valid_dir:  Path
    output_dir: Path
    metrics:    TrainMetrics
    resume_dir: Path | None = None

In [7]:
import os
from core.constants import *
from core import read_yaml, create_directories

class ConfigurationManager:
    def __init__(self, 
                 config_filepath=CONFIG_FILE_PATH, 
                 params_filepath=PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])
        
    def get_train_config(self) -> TrainConfig:
        config = self.config.train_config
        params = self.params.train_params
        create_directories([config.output_dir])

        return TrainConfig(
            train_dir   = Path(config.train_dir),
            valid_dir   = Path(config.valid_dir),
            output_dir  = Path(config.output_dir),
            resume_dir  = Path(config.resume_dir) if config.resume_dir else None,
            metrics     = TrainMetrics(
                model_name      = params.model_name,
                image_size      = params.image_size,
                epochs          = params.epochs,
                batch_size      = params.batch_size,
                lr              = params.lr,
                weight_decay    = params.weight_decay,
                unfreeze_epoch  = params.unfreeze_epoch,
                top_k_checkpoints = params.top_k_checkpoints,
                early_stopping  = params.early_stopping_patience
            )
        )

In [8]:
import numpy as np
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms

def get_transforms(augment: bool):
    if augment:
        return transforms.Compose([
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(15),
            transforms.ColorJitter(brightness=0.2, contrast=0.2),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])
    return transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])


def build_dataloaders(train_dir, valid_dir, batch_size: int, num_workers: int = 2):
    train_ds = datasets.ImageFolder(train_dir, transform=get_transforms(augment=True))
    val_ds   = datasets.ImageFolder(valid_dir, transform=get_transforms(augment=False))

    targets      = np.array(train_ds.targets)
    class_counts = np.bincount(targets)
    weights      = 1.0 / class_counts[targets]
    sampler      = WeightedRandomSampler(weights, len(weights), replacement=True)

    train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=sampler, num_workers=num_workers)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False,   num_workers=num_workers)

    return train_loader, val_loader, len(train_ds.classes), class_counts

In [9]:
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

def build_optimizer(model, lr: float, freeze: bool, config):
    params = filter(lambda p: p.requires_grad, model.parameters()) if freeze else model.parameters()
    actual_lr = lr if freeze else lr * 0.1
    optimizer = AdamW(params, lr=actual_lr, weight_decay=config.metrics.weight_decay)
    scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2)
    return optimizer, scheduler

def build_criterion(class_counts, num_classes: int, device, config):
    cw = torch.FloatTensor(1.0 / class_counts).to(device)
    cw = cw / cw.sum() * num_classes
    return nn.CrossEntropyLoss(weight=cw, label_smoothing=0.1)

In [10]:
import os
import json
import torch
from tqdm import tqdm
import heapq
import boto3

from torch.amp import autocast

from core.logger import logger
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()

def run_epoch(model, loader, criterion, optimizer, device, train: bool, scaler=None) -> dict:
    model.train() if train else model.eval()
    total_loss, total_correct, total_n = 0, 0, 0

    with torch.set_grad_enabled(train):
        for imgs, labels in tqdm(loader, desc="Train" if train else "Valid", leave=False):
            imgs, labels = imgs.to(device), labels.to(device)

            with autocast(device_type="cuda", dtype=torch.float16):
                out  = model(imgs)
                loss = criterion(out, labels)

            if train:
                optimizer.zero_grad()
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

            total_loss    += loss.item() * imgs.size(0)
            total_correct += (out.argmax(1) == labels).sum().item()
            total_n       += imgs.size(0)

    return {"loss": total_loss / total_n, "acc": total_correct / total_n}

@torch.no_grad()
def run_validate(model, loader, criterion, device) -> dict:
    return run_epoch(model, loader, criterion, optimizer=None, device=device, train=False)

def save_model_metadata(output_dir: Path, class_names: list[str], image_size: int):
    metadata = {
        "class_names": class_names,
        "num_classes": len(class_names),
        "image_size": image_size,
    }
    with open(output_dir / "metadata.json", "w") as f:
        json.dump(metadata, f, indent=4)

def push_to_s3(local_dir: Path, prefix: str = "train"):
    s3 = boto3.client(
        "s3",
        aws_access_key_id     = os.getenv("AWS_ACCESS_KEY_ID"),
        aws_secret_access_key = os.getenv("AWS_SECRET_ACCESS_KEY"),
        region_name           = os.getenv("AWS_REGION"),
    )
    bucket = os.getenv("S3_BUCKET")

    for filepath in local_dir.rglob("*"):
        if filepath.is_file():
            key = f"{prefix}/{filepath.relative_to(local_dir)}"
            s3.upload_file(str(filepath), bucket, key)
            logger.info(f"Uploaded: {key}")

class CheckpointManager:
    def __init__(self, output_dir: Path, top_k: int = 3):
        self.output_dir = output_dir
        self.top_k = top_k
        self.heap = []
        os.makedirs(self.output_dir, exist_ok=True)

    def save_if_best(self, trainer, val_acc: float, epoch: int):
        filepath = self.output_dir / f"epoch{epoch:03d}_acc{val_acc:.4f}.pth"

        if len(self.heap) < self.top_k:
            trainer.save_checkpoint(filepath, epoch)
            heapq.heappush(self.heap, (val_acc, epoch, filepath))
            logger.info(f"  ✓ Saved checkpoint ({len(self.heap)}/{self.top_k}): {filepath.name}")
            return

        if val_acc > self.heap[0][0]:
            trainer.save_checkpoint(filepath, epoch)
            heapq.heappush(self.heap, (val_acc, epoch, filepath))

            _, _, worst_path = heapq.heappop(self.heap)
            if worst_path.exists():
                worst_path.unlink()
            logger.info(f"  ✓ New checkpoint kept, removed: {worst_path.name}")

    def get_best_checkpoint(self) -> Path:
        _, _, best_path = max(self.heap)
        return best_path

In [11]:
import torch
from pathlib import Path
import numpy as np
from torch.amp import GradScaler
from core.logger import logger

class Trainer:
    def __init__(self, model: BaseVisionModel, class_counts: np.ndarray, config):
        self.config = config
        self.best_val_acc = 0.0
        self.epochs_without_improvement = 0
        
        self.model = model
        self.scaler = GradScaler()
        self.optimizer, self.scheduler = build_optimizer(model.model, config.metrics.lr, freeze=True, config=config)
        self.criterion = build_criterion(class_counts, model.num_classes, model.device, config=config)
        self.checkpoint_mgr = CheckpointManager(model.checkpoint_dir, top_k=config.metrics.top_k_checkpoints)

    def _unfreeze_backbone(self):
        for p in self.model.model.features.parameters():
            p.requires_grad = True
        self.optimizer, self.scheduler = build_optimizer(self.model.model, self.config.metrics.lr, freeze=False, config=self.config)

    def fit(self, train_loader, val_loader) -> None:
        start_epoch = 1

        if self.config.resume_dir and self.config.resume_dir.exists():
            last_checkpoint = self.config.resume_dir / "last_checkpoint.pth"
            if last_checkpoint.exists():
                loaded_epoch, self.best_val_acc = self.load_checkpoint(last_checkpoint)
                start_epoch = loaded_epoch + 1
                logger.info(f"Resumed from epoch {start_epoch-1}, best_val_acc={self.best_val_acc:.4f}")

        for epoch in range(start_epoch, self.config.metrics.epochs + 1):
            if epoch == self.config.metrics.unfreeze_epoch:
                self._unfreeze_backbone()
                print("!! Backbone unfrozen !!")

            train_m = run_epoch(self.model.model, train_loader, self.criterion, self.optimizer, self.model.device, train=True, scaler=self.scaler)
            self.scheduler.step()
            val_m = run_validate(self.model.model, val_loader, self.criterion, self.model.device)

            self.checkpoint_mgr.save_if_best(self, val_m["acc"], epoch)
            self.save_checkpoint(self.model.checkpoint_dir / "last_checkpoint.pth", epoch)

            print(f"Epoch {epoch:02d}/{self.config.metrics.epochs}  loss={train_m['loss']:.4f}  train_acc={train_m['acc']:.4f}  val_acc={val_m['acc']:.4f}")

            if val_m["acc"] > self.best_val_acc:
                self.best_val_acc = val_m["acc"]
                self.epochs_without_improvement = 0
                print(f"  ^^^ New best! val_acc={val_m['acc']:.4f} ^^^")
            else:
                self.epochs_without_improvement += 1
            
            if self.epochs_without_improvement >= self.config.metrics.early_stopping:
                logger.info(f"Early stopping triggered at epoch {epoch} (no improvement for {self.config.metrics.early_stopping} epochs)")
                break

    def save_checkpoint(self, path: Path, epoch: int) -> None:
        torch.save({
            "epoch": epoch,
            "model_state": self.model.model.state_dict(),
            "optimizer_state": self.optimizer.state_dict(),
            "scheduler_state": self.scheduler.state_dict(),
            "best_val_acc": self.best_val_acc,
        }, path)

    def load_checkpoint(self, path: Path) -> tuple[int, float]:
        checkpoint = torch.load(path, map_location=self.model.device)
        self.model.model.load_state_dict(checkpoint["model_state"])
        self.optimizer.load_state_dict(checkpoint["optimizer_state"])
        self.scheduler.load_state_dict(checkpoint["scheduler_state"])
        return checkpoint["epoch"], checkpoint["best_val_acc"]

In [12]:
import sys
import torch
from core.logger import logger
from core.exception import CustomException

class Train:
    def __init__(self, config: TrainConfig):
        self.config = config

    def run(self):
        try:
            train_loader, val_loader, num_classes, class_counts = build_dataloaders(
                self.config.train_dir, self.config.valid_dir, self.config.metrics.batch_size
            )
            logger.info(f"Train: {len(train_loader.dataset):,}  Val: {len(val_loader.dataset):,}  Classes: {num_classes}")

            model = ModelFactory.create(self.config.metrics.model_name, self.config, num_classes)
            trainer = Trainer(model, class_counts, self.config)
            trainer.fit(train_loader, val_loader)

            best_checkpoint = trainer.checkpoint_mgr.get_best_checkpoint()
            trainer.load_checkpoint(best_checkpoint)

            sample_input = torch.randn(1, 3, self.config.metrics.image_size, self.config.metrics.image_size).to(model.device)
            onnx_path = model.checkpoint_dir / "weights.onnx"
            model.save_onnx(onnx_path, sample_input)

            save_model_metadata(
                output_dir=model.checkpoint_dir,
                class_names=train_loader.dataset.classes,
                image_size=self.config.metrics.image_size
            )

            push_to_s3(model.checkpoint_dir, prefix=f"train/{self.config.metrics.model_name}")

            logger.info(f"Training complete. Best val_acc={trainer.best_val_acc:.4f}")

        except Exception as e:
            raise CustomException(e, sys)

In [13]:
try:
    cfg_manager = ConfigurationManager()
    train_cfg   = cfg_manager.get_train_config()
    train_cls   = Train(config=train_cfg)
    train_cls.run()
except Exception as e:
    raise CustomException(e, sys)

[2026-06-19 04:38:05,219: INFO: __init__: yaml file: config/config.yaml loaded successfully]
[2026-06-19 04:38:05,224: INFO: __init__: yaml file: params.yaml loaded successfully]
[2026-06-19 04:38:05,226: INFO: __init__: created directory at: artifacts]
[2026-06-19 04:38:05,227: INFO: __init__: created directory at: artifacts/train]
[2026-06-19 04:38:05,284: INFO: 378157631: Train: 19,007  Val: 4,073  Classes: 10]


[2026-06-19 04:38:55,205: INFO: 1356554110:   ✓ Saved checkpoint (1/3): epoch001_acc0.5669.pth]


Epoch 01/20  loss=1.6062  train_acc=0.4765  val_acc=0.5669
  ^^^ New best! val_acc=0.5669 ^^^


[2026-06-19 04:39:33,128: INFO: 1356554110:   ✓ Saved checkpoint (2/3): epoch002_acc0.5711.pth]
Epoch 02/20  loss=1.4839  train_acc=0.5427  val_acc=0.5711
  ^^^ New best! val_acc=0.5711 ^^^


[2026-06-19 04:40:11,467: INFO: 1356554110:   ✓ Saved checkpoint (3/3): epoch003_acc0.5738.pth]
Epoch 03/20  loss=1.4583  train_acc=0.5558  val_acc=0.5738
  ^^^ New best! val_acc=0.5738 ^^^


...

Epoch 16/20  loss=0.6990  train_acc=0.9217  val_acc=0.7982


[2026-06-19 04:54:42,404: INFO: 1356554110:   ✓ New checkpoint kept, removed: epoch014_acc0.7997.pth]
Epoch 17/20  loss=0.6805  train_acc=0.9321  val_acc=0.8102
  ^^^ New best! val_acc=0.8102 ^^^


[2026-06-19 04:55:44,113: INFO: 1356554110:   ✓ New checkpoint kept, removed: epoch012_acc0.8004.pth]
Epoch 18/20  loss=0.6604  train_acc=0.9403  val_acc=0.8110
  ^^^ New best! val_acc=0.8110 ^^^


[2026-06-19 04:56:45,439: INFO: 1356554110:   ✓ New checkpoint kept, removed: epoch013_acc0.8051.pth]
Epoch 19/20  loss=0.6424  train_acc=0.9485  val_acc=0.8191
  ^^^ New best! val_acc=0.8191 ^^^


[2026-06-19 04:57:50,575: INFO: 1356554110:   ✓ New checkpoint kept, removed: epoch017_acc0.8102.pth]
Epoch 20/20  loss=0.6245  train_acc=0.9548  val_acc=0.8141
[2026-06-19 04:58:02,741: INFO: 1356554110: Uploaded: train/efficientnet/epoch018_acc0.8110.pth]
[2026-06-19 04:58:06,217: INFO: 1356554110: Uploaded: train/efficientnet/epoch020_acc0.8141.pth]
[2026-06-19 04:58:06,495: INFO: 1356554110: Uploaded: train/efficientnet/metadata.json]
[2026-06-19 04:58:08,940: INFO: 1356554110: Uploaded: train/efficientnet/epoch019_acc0.8191.pth]
[2026-06-19 04:58:11,445: INFO: 1356554110: Uploaded: train/efficientnet/last_checkpoint.pth]
[2026-06-19 04:58:14,298: INFO: 1356554110: Uploaded: train/efficientnet/weights.onnx]
[2026-06-19 04:58:14,299: INFO: 378157631: Training complete. Best val_acc=0.8191]
